# JAX-CrossCat GPU Benchmark

This notebook benchmarks the packed v2 Gibbs sweep kernels against the unpacked
and packed v1 implementations. Run on Google Colab with a GPU runtime for best
results.

In [ ]:
# Install and import
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "-q"])

import time

import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")

In [ ]:
from crosscat.gibbs import gibbs_sweep
from crosscat.model import initialize
from crosscat.packed import pack_state, packed_gibbs_sweep, unpack_state
from crosscat.synthetic import generate_crosscat_data
from crosscat.types import ColumnType


def make_dataset(key, n_rows, n_cols):
    types = [ColumnType.CONTINUOUS] * (n_cols // 2) + [ColumnType.CATEGORICAL] * (
        n_cols - n_cols // 2
    )
    result = generate_crosscat_data(key, n_rows, types, n_views=2, n_clusters=3)
    return result["data"], types


key = jax.random.key(42)
sizes = {"small": (100, 10), "medium": (500, 20)}
datasets = {}
for name, (nr, nc) in sizes.items():
    key, k = jax.random.split(key)
    data, types = make_dataset(k, nr, nc)
    key, k = jax.random.split(key)
    state = initialize(k, data, types)
    packed = pack_state(state)
    datasets[name] = {"data": data, "types": types, "state": state, "packed": packed}
    print(f"{name}: {nr}x{nc}")

In [ ]:
results = {}
for name, ds in datasets.items():
    key = jax.random.key(100)
    # Unpacked (original Python-loop kernels)
    start = time.time()
    gibbs_sweep(key, ds["state"], ds["data"], n_sweeps=1)
    t_unpacked = time.time() - start

    # Packed (JIT-compiled, include compile time)
    start = time.time()
    packed_gibbs_sweep(key, ds["packed"], ds["data"], n_sweeps=1)
    t_packed_cold = time.time() - start

    # Packed (post-JIT, warm)
    key2 = jax.random.key(101)
    start = time.time()
    packed_gibbs_sweep(key2, ds["packed"], ds["data"], n_sweeps=1)
    t_packed_warm = time.time() - start

    results[name] = {
        "unpacked": t_unpacked,
        "packed_cold": t_packed_cold,
        "packed_warm": t_packed_warm,
    }
    print(f"\n{name}:")
    print(f"  Unpacked:       {t_unpacked:.2f}s")
    print(f"  Packed (cold):  {t_packed_cold:.2f}s")
    print(f"  Packed (warm):  {t_packed_warm:.2f}s")

In [ ]:
from crosscat.validate import validate_state

key = jax.random.key(200)
ds = datasets["small"]
packed_result = packed_gibbs_sweep(key, ds["packed"], ds["data"], n_sweeps=2)
recovered = unpack_state(packed_result, ds["types"])
errors = validate_state(recovered, ds["data"])
print(f"Validation: {'PASS' if not errors else 'FAIL: ' + str(errors)}")

from crosscat.model import log_joint

lj = float(log_joint(recovered, ds["data"]))
print(f"Log joint: {lj:.2f} (finite: {jnp.isfinite(jnp.array(lj))})")